# Traffic Sign Classifier — 7-Epoch Reproducible Training + Submission

ImageNet pretrained EfficientNet-B0에서 시작해 7 epoch를 직접 fine-tuning합니다. 검증된 결과와 동일하게 cosine schedule 주기는 20으로 유지하며, 학습 직후 best checkpoint로 `result.csv`와 행 순서가 동일한 `result_confidence.csv`를 생성·검증합니다.

## 제출 규칙 체크리스트

- 예측 결과는 integer로 저장
- 제공된 정답지의 행/데이터 순서를 절대 변경하지 않음
- 최종 제출 CSV에는 header와 index를 추가하지 않음
- 제출 전 모든 셀을 처음부터 다시 실행하여 오류 확인
- 파일명은 안내된 team number / 제출 번호 형식을 따름
- 허용된 학습 데이터와 pretrained model만 사용하며, 규칙에 없는 우회·꼼수는 사용하지 않음

In [ ]:
from pathlib import Path
import os, random, zipfile, time, copy
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0
try:
    from torchvision.models import EfficientNet_B0_Weights
except ImportError:
    EfficientNet_B0_Weights = None  # torchvision 0.11 / CUDA 10.2 environment

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Reproducibility
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
set_seed()
REQUIRE_CUDA = True  # RTX 2060 local GPU training
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA GPU를 찾지 못했습니다. NVIDIA 드라이버를 업데이트하고 CUDA용 PyTorch를 설치한 뒤 '
        '커널을 재시작하세요. 확인: torch.cuda.is_available() == True'
    )
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'Device: {props.name} | VRAM: {props.total_memory / 2**30:.1f} GB | CUDA: {torch.version.cuda}')
else:
    print('Device: CPU')

## 경로 자동 탐색

현재 작업 폴더, 노트북 폴더 후보 및 하위 폴더에서 `Train/`과 `Test/`가 함께 있는 실제 데이터 루트를 찾습니다. 현재 제공 파일처럼 `data/data 2/`로 중첩되어도 동작합니다.

In [ ]:
def find_project_root():
    starts = [Path.cwd().resolve()]
    for start in starts:
        for p in [start, *start.parents]:
            if (p / 'cnn-trafficsign-torch-skeleton.ipynb').exists() or (p / 'data.zip').exists():
                return p
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
ZIP_PATH = PROJECT_ROOT / 'data.zip'
EXTRACT_ROOT = PROJECT_ROOT / 'data'
RESULT_TEMPLATE = PROJECT_ROOT / 'result_template_before_prediction.csv'
if not RESULT_TEMPLATE.exists():
    RESULT_TEMPLATE = PROJECT_ROOT / 'result.csv'
CHECKPOINT_PATH = PROJECT_ROOT / 'best_traffic_sign_efficientnet_b0_epoch7.pt'

def find_data_root(project_root):
    candidates = [project_root, project_root / 'data']
    candidates += [p for p in (project_root / 'data').rglob('*') if p.is_dir()] if (project_root / 'data').exists() else []
    for p in candidates:
        if (p / 'Train').is_dir() and (p / 'Test').is_dir():
            return p.resolve()
    return None

DATA_ROOT = find_data_root(PROJECT_ROOT)
if DATA_ROOT is None and ZIP_PATH.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        base = EXTRACT_ROOT.resolve()
        for member in zf.infolist():
            target = (base / member.filename).resolve()
            if base != target and base not in target.parents:
                raise RuntimeError(f'Unsafe zip member: {member.filename}')
        zf.extractall(base)
    DATA_ROOT = find_data_root(PROJECT_ROOT)
if DATA_ROOT is None:
    raise FileNotFoundError(f'Train/Test 폴더를 찾지 못했습니다. project={PROJECT_ROOT}, zip={ZIP_PATH}')
TRAIN_DIR, TEST_DIR = DATA_ROOT / 'Train', DATA_ROOT / 'Test'
print('Project root:', PROJECT_ROOT)
print('Data root:', DATA_ROOT)
print('Train/Test:', TRAIN_DIR, TEST_DIR)

In [ ]:
# Configuration: BATCH_SIZE may be lowered to 32 on a small GPU.
NUM_CLASSES = 43
IMAGE_SIZE = 128  # source images are 15..250px; avoids wasteful 224px upscaling
BATCH_SIZE = 96 if torch.cuda.is_available() else 16  # RTX 2060 6GB-safe default
NUM_WORKERS = 0 if os.name == 'nt' else min(4, os.cpu_count() or 0)  # stable Windows/Jupyter loading
EPOCHS = 7
SCHEDULER_T_MAX = 20  # preserve the verified 20-epoch cosine schedule through epoch 7
PATIENCE = 7  # evaluate all seven epochs
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.15

weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if EfficientNet_B0_Weights is not None else None
mean, std = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomApply([transforms.RandomAffine(12, translate=(0.08, 0.08), scale=(0.85, 1.15), shear=5)], p=0.8),
    transforms.RandomPerspective(distortion_scale=0.12, p=0.25),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.18, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(p=0.12, scale=(0.02, 0.08), ratio=(0.5, 2.0)),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [ ]:
# torchvision's default lexical sort would map 10 before 2; force numeric folder order.
class NumericImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = sorted([e.name for e in os.scandir(directory) if e.is_dir()], key=int)
        return classes, {name: int(name) for name in classes}

base_ds = NumericImageFolder(TRAIN_DIR)
expected_classes = [str(i) for i in range(NUM_CLASSES)]
if base_ds.classes != expected_classes:
    raise ValueError(f'Unexpected classes: {base_ds.classes}')
targets = np.asarray(base_ds.targets)
indices = np.arange(len(base_ds))
train_idx, val_idx = train_test_split(indices, test_size=VAL_RATIO, random_state=SEED, stratify=targets)
train_full = NumericImageFolder(TRAIN_DIR, transform=train_tf)
val_full = NumericImageFolder(TRAIN_DIR, transform=eval_tf)
train_ds, val_ds = Subset(train_full, train_idx), Subset(val_full, val_idx)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, persistent_workers=NUM_WORKERS > 0, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, persistent_workers=NUM_WORKERS > 0, **loader_args)
counts = np.bincount(targets[train_idx], minlength=NUM_CLASSES)
# Mild inverse-sqrt weighting improves minority classes without unstable full oversampling.
class_weights = np.sqrt(counts.sum() / (NUM_CLASSES * counts))
class_weights = torch.tensor(class_weights / class_weights.mean(), dtype=torch.float32, device=DEVICE)
print(f'train={len(train_ds):,}, val={len(val_ds):,}, classes={len(base_ds.classes)}')
print('class count range:', counts.min(), counts.max())

In [ ]:
# Old CUDA 10.2/Python environments can fail on the Windows certificate store.
import certifi, ssl
os.environ['SSL_CERT_FILE'] = certifi.where()
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
torch.hub.set_dir(str(PROJECT_ROOT / '.torch_cache'))
model = efficientnet_b0(weights=weights) if weights is not None else efficientnet_b0(pretrained=True)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(p=0.30), nn.Linear(in_features, NUM_CLASSES))
model.to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCHEDULER_T_MAX, eta_min=LR / 50)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')
print(f'parameters={sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss, all_y, all_pred = 0.0, [], []
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * y.size(0)
            all_y.extend(y.detach().cpu().tolist())
            all_pred.extend(logits.argmax(1).detach().cpu().tolist())
    return {
        'loss': total_loss / len(loader.dataset),
        'acc': accuracy_score(all_y, all_pred),
        'macro_f1': f1_score(all_y, all_pred, average='macro'),
    }

best_f1, stale, history = -1.0, 0, []
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_m = run_epoch(train_loader, True)
    val_m = run_epoch(val_loader, False)
    scheduler.step()
    row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_m.items()}, **{f'val_{k}': v for k, v in val_m.items()}}
    history.append(row)
    print(f"{epoch:02d}/{EPOCHS} | train loss {train_m['loss']:.4f} f1 {train_m['macro_f1']:.4f} | val loss {val_m['loss']:.4f} acc {val_m['acc']:.4f} f1 {val_m['macro_f1']:.4f} | {time.time()-t0:.0f}s")
    if val_m['macro_f1'] > best_f1:
        best_f1, stale = val_m['macro_f1'], 0
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_macro_f1': best_f1, 'classes': base_ds.classes}, CHECKPOINT_PATH)
    else:
        stale += 1
        if stale >= PATIENCE:
            print('Early stopping')
            break
print('Best validation macro-F1:', best_f1, '| checkpoint:', CHECKPOINT_PATH)

In [ ]:
# Final checkpoint verification
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model'])
final_val = run_epoch(val_loader, False)
print('loaded epoch:', checkpoint['epoch'], 'saved F1:', checkpoint['val_macro_f1'])
print('verified validation:', final_val)
pd.DataFrame(history).round(5)

## Test inference and CSV creation

학습 직후 로드된 best checkpoint로 Test를 추론합니다. 빈 template 백업의 파일명 순서를 그대로 유지한 `result.csv`와 동일한 행 순서의 confidence 파일을 생성합니다.

In [ ]:
class OrderedTestDataset(Dataset):
    def __init__(self, test_dir, names, transform):
        self.test_dir, self.names, self.transform = Path(test_dir), list(names), transform
        missing = [n for n in self.names if not (self.test_dir / n).is_file()]
        if missing:
            raise FileNotFoundError(f'{len(missing)} test files missing; first={missing[0]}')
    def __len__(self): return len(self.names)
    def __getitem__(self, idx):
        with Image.open(self.test_dir / self.names[idx]) as image:
            return self.transform(image.convert('RGB'))

RUN_INFERENCE = True
if RUN_INFERENCE:
    if not RESULT_TEMPLATE.exists():
        raise FileNotFoundError(f'Result template missing: {RESULT_TEMPLATE}')
    template = pd.read_csv(RESULT_TEMPLATE, dtype=str)
    if template.shape[1] < 1:
        raise ValueError('result.csv has no filename column')
    output_path = PROJECT_ROOT / 'result.csv'
    confidence_output_path = PROJECT_ROOT / 'result_confidence.csv'
    # Abort before inference if Excel/another program has either output locked.
    for target in (output_path, confidence_output_path):
        if target.exists():
            try:
                with target.open('r+', encoding='utf-8'):
                    pass
            except PermissionError as exc:
                raise PermissionError(f'출력 파일이 다른 프로그램에서 열려 있습니다: {target}') from exc
    test_names = template.iloc[:, 0].tolist()  # exact provided order
    test_ds = OrderedTestDataset(TEST_DIR, test_names, eval_tf)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    predictions, confidences = [], []
    model.eval()
    with torch.inference_mode():
        for x in test_loader:
            x = x.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                probabilities = torch.softmax(model(x), dim=1)
                confidence, prediction = probabilities.max(dim=1)
                predictions.extend(prediction.cpu().tolist())
                confidences.extend(confidence.cpu().tolist())
    assert len(predictions) == len(test_names)
    assert all(isinstance(v, int) and 0 <= v < NUM_CLASSES for v in predictions)
    assert len(confidences) == len(test_names)
    pd.DataFrame({0: test_names, 1: np.asarray(predictions, dtype=np.int64)}).to_csv(output_path, index=False, header=False)
    confidence_frame = pd.DataFrame({
        'original_row': np.arange(len(test_names), dtype=np.int64),
        'filename': test_names,
        'predicted_class': np.asarray(predictions, dtype=np.int64),
        'confidence': np.asarray(confidences, dtype=np.float64),
    })
    confidence_frame.to_csv(confidence_output_path, index=False, float_format='%.8f')
    print('Saved:', output_path, 'and', confidence_output_path, '| rows:', len(predictions))
else:
    print('Inference skipped.')

## 최종 violation 재점검

Inference를 실행한 뒤 아래 셀로 행 순서, integer 범위, header/index 부재를 다시 검사합니다.

In [ ]:
if RUN_INFERENCE:
    raw = pd.read_csv(output_path, header=None, dtype={0: str, 1: np.int64})
    assert raw.shape == (len(test_names), 2)
    assert raw.iloc[:, 0].tolist() == test_names, 'Row order changed'
    assert raw.iloc[:, 1].between(0, NUM_CLASSES - 1).all(), 'Invalid class'
    first_line = output_path.read_text(encoding='utf-8').splitlines()[0]
    assert first_line.split(',')[0] == test_names[0], 'Unexpected header/index'
    confidence_raw = pd.read_csv(confidence_output_path)
    assert confidence_raw.shape == (len(test_names), 4)
    assert confidence_raw['original_row'].tolist() == list(range(len(test_names)))
    assert confidence_raw['filename'].tolist() == test_names
    assert confidence_raw['predicted_class'].tolist() == raw.iloc[:, 1].tolist()
    assert confidence_raw['confidence'].between(0.0, 1.0).all()
    print('PASS: exact row order, integer classes, no submission header/index, aligned confidence rows.')
else:
    print('Final CSV check skipped until RUN_INFERENCE=True.')